# Knowledge Graph Extraction — Pipeline 3 Passes

**Stratégie de prompt engineering :**
- **Pass 1** : Extraction des types d'entités uniquement (labels)
- **Pass 2** : Extraction des entités avec leurs champs complets
- **Pass 3** : Extraction des relations entre entités connues

**Chunking** : par `=== DOCUMENT X ===` — chaque document est traité indépendamment, puis les résultats sont mergés.

---

## Bloc 1 — Imports et configuration

In [1]:
import os
import json
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA

load_dotenv()

print("Imports OK")
print(f"NVIDIA_API_KEY : {'✓' if os.getenv('NVIDIA_API_KEY') else '✗ MANQUANTE'}")
print(f"NEO4J_URI      : {os.getenv('NEO4J_URI', 'Non défini')}")

Imports OK
NVIDIA_API_KEY : ✓
NEO4J_URI      : bolt://localhost:7687


## Bloc 2 — Initialisation du LLM

In [2]:
llm = ChatNVIDIA(
    model="mistralai/mistral-large-3-675b-instruct-2512",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0,
    max_completion_tokens=32000,
)

## Bloc 3 — Schéma du graphe

In [3]:
SCHEMA = """
NODES (entity: fields):
  Company         : name, activity, certifications
  Supplier        : name
  Agreement       : name, description, type, startdate, enddate
  Document        : reference, title, version, date, owner
  Clients         : name, sector
  Clause          : name, description
  governance_body : name, acronyme, role
  Claim           : claim_type, value, unit, scope, source_doc, source_doc_id, quote
  Roles           : title
  Asset           : type, description, classification
  Algorithm       : name, use
  Protocol        : name, use, source
  Risk            : description, level
  Framework       : name, type, use
  Control         : name, requirement, source, Cycle
  Technology      : name, use, source

RELATIONSHIPS (source -[REL]-> target):
  Agreement       -[for]->              Supplier
  Company         -[Use]->              Agreement
  Supplier        -[HAS]->              Company
  Company         -[HAS]->              Document
  Company         -[HAS]->              Clients
  Company         -[HAS]->              governance_body
  Company         -[has_requirement]->  Clause
  Supplier        -[apply_to]->         Clause
  governance_body -[supervise]->        Roles
  governance_body -[approves]->         Claim
  Claim           -[Concerns]->         Asset
  Claim           -[Asserted_by]->      Roles
  Roles           -[operates]->         Technology
  Roles           -[EXECUTES]->         Control
  Algorithm       -[protects]->         Asset
  Protocol        -[used_by]->          Algorithm
  Protocol        -[mitigates]->        Risk
  Risk            -[targeting]->        Asset
  Framework       -[required_by]->      Control
  Control         -[IMPLEMENTED_BY]->   Technology
  Technology      -[Has_access_to]->    Asset
"""

NODE_LABELS = [
    "Company", "Supplier", "Agreement", "Document", "Clients",
    "Clause", "governance_body", "Claim", "Roles", "Asset",
    "Algorithm", "Protocol", "Risk", "Framework", "Control", "Technology"
]

print(f"Schéma défini : {len(NODE_LABELS)} types de nœuds, 21 types de relations")

Schéma défini : 16 types de nœuds, 21 types de relations


## Bloc 4 — Prompts des 3 passes

**Stratégie de prompt engineering :**
- Pass 1 = tâche simple → liste les labels présents dans le chunk
- Pass 2 = tâche ciblée → extrait uniquement les labels identifiés en pass 1 avec tous leurs champs
- Pass 3 = tâche relationnelle → connaissant les entités, extrait les relations entre elles

In [4]:
# ─────────────────────────────────────────────────────────────────
# PASS 1 — Quels types d'entités sont présents dans ce chunk ?
# ─────────────────────────────────────────────────────────────────
PROMPT_PASS1 = f"""
You are a knowledge-graph analyst. Your task is to SCAN a document chunk
and identify which entity types from the schema are mentioned.

Schema node labels:
{', '.join(NODE_LABELS)}

Rules:
- Return ONLY a JSON array of label strings that are present in the text.
- Include a label only if at least one instance is clearly identifiable.
- No explanation, no markdown, no extra text.

Output format example:
["Company", "Document", "Roles", "Control"]
"""

# ─────────────────────────────────────────────────────────────────
# PASS 2 — Extraire les entités avec leurs champs (labels ciblés)
# ─────────────────────────────────────────────────────────────────
def build_prompt_pass2(detected_labels: list[str]) -> str:
    # Filtre le schéma pour ne montrer que les types détectés → prompt plus court et plus précis
    schema_lines = []
    for line in SCHEMA.strip().split("\n"):
        for label in detected_labels:
            if line.strip().startswith(label):
                schema_lines.append(line)
                break
    focused_schema = "\n".join(schema_lines)

    return f"""
You are a knowledge-graph extraction assistant.
Extract ALL entities of the following types from the document chunk.
Be exhaustive — extract every instance you can find.

Target entity types and their fields:
{focused_schema}

Rules:
- Use ONLY the node labels listed above.
- Extract every field you can find; omit fields that are not mentioned.
- String values only, no nested objects.
- Return ONLY a valid JSON array, no markdown, no explanation.

Output format:
[
  {{"label": "<NodeLabel>", "properties": {{"field": "value"}}}},
  ...
]
"""

# ─────────────────────────────────────────────────────────────────
# PASS 3 — Extraire les relations entre entités connues
# ─────────────────────────────────────────────────────────────────
def build_prompt_pass3(entities: list[dict]) -> str:
    # Résumé compact des entités déjà extraites → ancrage pour la relation
    entity_summary = json.dumps(
        [{"label": e["label"], "key": list(e["properties"].items())[:1]} for e in entities],
        ensure_ascii=False
    )
    return f"""
You are a knowledge-graph relation extractor.
Given the document chunk and the list of already-extracted entities below,
identify all relationships between them that match the schema.

Schema relationships:
{SCHEMA.split('RELATIONSHIPS')[1].strip()}

Already extracted entities (label + identifying key):
{entity_summary}

Rules:
- Use ONLY relationship types from the schema.
- from_key / to_key must match an entity from the list above.
- Return ONLY a valid JSON array, no markdown, no explanation.

Output format:
[
  {{
    "from_label": "<NodeLabel>",
    "from_key":   {{"field": "value"}},
    "rel_type":   "<REL_TYPE>",
    "to_label":   "<NodeLabel>",
    "to_key":     {{"field": "value"}}
  }},
  ...
]
"""

print("Prompts des 3 passes définis")

Prompts des 3 passes définis


## Bloc 5 — Fonctions utilitaires

In [10]:
def call_llm(system_prompt: str, user_text: str) -> str:
    """Appelle le LLM et retourne le contenu brut."""
    messages = [
        ("system", system_prompt),
        ("human", f"Document chunk:\n\n{user_text}"),
    ]
    raw = llm.invoke(messages).content.strip()
    # Nettoyage des balises markdown
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    return raw


def parse_json(raw: str, context: str = ""):
    """Parse JSON avec gestion d'erreur explicite."""
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"      [JSON ERROR] {context} : {e}")
        return None


def split_by_document(text: str, marker: str = "=== DOCUMENT") -> list[dict]:
    chunks = []
    parts = text.split(marker)
    for part in parts[1:]:
        first_line = part.split("\n")[0].strip().rstrip("===").strip()
        doc_id = f"DOCUMENT {first_line}"
        content = (marker + part).strip()
        
        # Ignorer les chunks trop courts (= table des matières)
        if len(content) < 200:
            continue
            
        chunks.append({"id": doc_id, "text": content})
    return chunks if chunks else [{"id": "FULL", "text": text}]


def merge_results(results: list[dict]) -> dict:
    """Fusionne les résultats de tous les chunks sans doublons."""
    merged_entities = []
    merged_relations = []
    seen_entities = set()
    seen_relations = set()

    for result in results:
        for entity in result.get("entities", []):
            key = (entity["label"], str(entity.get("properties", {})))
            if key not in seen_entities:
                seen_entities.add(key)
                merged_entities.append(entity)
        for rel in result.get("relationships", []):
            key = (
                rel.get("from_label"),
                str(rel.get("from_key")),
                rel.get("rel_type"),
                rel.get("to_label"),
                str(rel.get("to_key")),
            )
            if key not in seen_relations:
                seen_relations.add(key)
                merged_relations.append(rel)

    return {"entities": merged_entities, "relationships": merged_relations}


print("Fonctions utilitaires prêtes")

Fonctions utilitaires prêtes


## Bloc 6 — Pipeline 3 passes pour un chunk

In [11]:
def extract_chunk_3pass(chunk: dict, verbose: bool = True) -> dict:
    """
    Applique les 3 passes d'extraction sur un chunk de document.

    Pass 1 : Détection des types d'entités présents
    Pass 2 : Extraction des entités avec champs complets
    Pass 3 : Extraction des relations entre entités

    Retourne : {"entities": [...], "relationships": [...]}
    """
    doc_id = chunk["id"]
    text   = chunk["text"]

    if verbose:
        print(f"\n   [{doc_id}] ({len(text):,} chars)")

    # ── PASS 1 : Quels types d'entités sont présents ? ────────────
    if verbose:
        print(f"      Pass 1 — Détection des labels...", end=" ", flush=True)

    raw1 = call_llm(PROMPT_PASS1, text)
    detected_labels = parse_json(raw1, f"{doc_id} Pass1")

    if not detected_labels or not isinstance(detected_labels, list):
        if verbose:
            print("  Aucun label détecté, chunk ignoré")
        return {"entities": [], "relationships": []}

    # Filtrer pour ne garder que les labels valides du schéma
    detected_labels = [l for l in detected_labels if l in NODE_LABELS]
    if verbose:
        print(f"{len(detected_labels)} labels : {detected_labels}")

    # ── PASS 2 : Extraction des entités avec champs ───────────────
    if verbose:
        print(f"      Pass 2 — Extraction des entités...", end=" ", flush=True)

    prompt2 = build_prompt_pass2(detected_labels)
    raw2 = call_llm(prompt2, text)
    entities = parse_json(raw2, f"{doc_id} Pass2")

    if not entities or not isinstance(entities, list):
        if verbose:
            print("⚠️  Aucune entité extraite")
        return {"entities": [], "relationships": []}

    if verbose:
        print(f"{len(entities)} entités")

    # ── PASS 3 : Extraction des relations ─────────────────────────
    if verbose:
        print(f"      Pass 3 — Extraction des relations...", end=" ", flush=True)

    prompt3 = build_prompt_pass3(entities)
    raw3 = call_llm(prompt3, text)
    relationships = parse_json(raw3, f"{doc_id} Pass3")

    if not relationships or not isinstance(relationships, list):
        if verbose:
            print("  Aucune relation extraite")
        relationships = []
    else:
        if verbose:
            print(f"{len(relationships)} relations")

    return {"entities": entities, "relationships": relationships}


print("Pipeline 3 passes prêt")

Pipeline 3 passes prêt


## Bloc 7 — Extraction complète par fichier

In [12]:
def extract_file_3pass(filepath: str) -> dict:
    """
    Charge un fichier, le découpe par document (=== DOCUMENT X ===),
    applique le pipeline 3 passes sur chaque chunk,
    et retourne le résultat mergé.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    chunks = split_by_document(text)
    print(f"   {len(chunks)} chunks (documents) trouvés")

    all_results = []
    for chunk in chunks:
        try:
            result = extract_chunk_3pass(chunk, verbose=True)
            all_results.append(result)
        except Exception as e:
            print(f"        Erreur sur {chunk['id']} : {e}")
            all_results.append({"entities": [], "relationships": []})

    merged = merge_results(all_results)
    print(f"\n    Total : {len(merged['entities'])} entités, {len(merged['relationships'])} relations")
    return merged


print("Fonction d'extraction complète prête")

Fonction d'extraction complète prête


## Bloc 8 — Lancement de l'extraction

In [13]:
DOCUMENTS = [
    "/home/maroua/Desktop/cassiope/SecuraOps High.txt",
    # "/home/maroua/Desktop/cassiope/SafeLink Contradictory.txt",
]

raw_results = {}

for filepath in DOCUMENTS:
    nom = filepath.split("/")[-1]
    print(f"\n{'='*60}")
    print(f"Extraction : {nom}")
    print(f"{'='*60}")
    try:
        result = extract_file_3pass(filepath)
        raw_results[filepath] = result
    except Exception as e:
        print(f"  Erreur globale : {e}")
        raw_results[filepath] = {"entities": [], "relationships": []}


Extraction : SecuraOps High.txt
   22 chunks (documents) trouvés

   [DOCUMENT 1 : Politique de Sécurité de l’Information] (9,640 chars)
      Pass 1 — Détection des labels... 11 labels : ['Company', 'Clients', 'Document', 'governance_body', 'Roles', 'Asset', 'Risk', 'Framework', 'Control', 'Technology', 'Clause']
      Pass 2 — Extraction des entités... 75 entités
      Pass 3 — Extraction des relations... 

KeyboardInterrupt: 

## Bloc 9 — Visualisation des résultats

In [ ]:
from collections import Counter

for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1]
    print(f"\n{nom}")
    print(f"   Entités   : {len(result['entities'])}")
    print(f"   Relations : {len(result['relationships'])}")
    node_counts = Counter(e['label'] for e in result['entities'])
    print("   Types d'entités :")
    for label, count in sorted(node_counts.items()):
        print(f"      {label:<20} : {count}")

## Bloc 10 — Sauvegarde JSON

In [ ]:
for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1].replace(".txt", "")
    output_file = f"{nom}_graph_3pass.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"Sauvegardé : {output_file}")

## Bloc 11 — Génération des requêtes Cypher

In [ ]:
def props_str(props: dict) -> str:
    parts = [f"{k}: {json.dumps(v)}" for k, v in props.items()]
    return "{" + ", ".join(parts) + "}"


def json_to_cypher(extracted: dict, company_label: str = None) -> list:
    statements = []
    for entity in extracted.get("entities", []):
        label = entity["label"]
        props = entity.get("properties", {})
        if props:
            if company_label:
                statements.append(f"MERGE (n:{label}:{company_label} {props_str(props)});")
            else:
                statements.append(f"MERGE (n:{label} {props_str(props)});")
    for rel in extracted.get("relationships", []):
        fl = rel["from_label"]
        fk = props_str(rel["from_key"])
        rt = rel["rel_type"]
        tl = rel["to_label"]
        tk = props_str(rel["to_key"])
        if company_label:
            statements.append(
                f"MATCH (a:{fl}:{company_label} {fk}), (b:{tl}:{company_label} {tk}) MERGE (a)-[:{rt}]->(b);"
            )
        else:
            statements.append(
                f"MATCH (a:{fl} {fk}), (b:{tl} {tk}) MERGE (a)-[:{rt}]->(b);"
            )
    return statements


company_map = {
    "SecuraOps High": "SecuraOps",
    "SafeLink Contradictory": "SafeLink",
}

cypher_by_file = {}
for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1].replace(".txt", "")
    company = company_map.get(nom, nom)
    stmts = json_to_cypher(result, company_label=company)
    cypher_by_file[nom] = stmts
    print(f"✅ {nom} : {len(stmts)} statements Cypher")

## Bloc 12 — Envoi vers Neo4j

In [ ]:
from neo4j import GraphDatabase

uri      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
user     = os.getenv("NEO4J_USER",     "neo4j")
password = os.getenv("NEO4J_PASSWORD", "")

driver = GraphDatabase.driver(uri, auth=(user, password))

# Vider la base
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("Base vidée")

# Envoi des statements
for nom, stmts in cypher_by_file.items():
    print(f"\nEnvoi de {nom} vers Neo4j...")
    errors = 0
    with driver.session() as session:
        for stmt in stmts:
            try:
                session.run(stmt)
            except Exception as e:
                errors += 1
    print(f"   {len(stmts) - errors} statements OK")
    if errors:
        print(f"   {errors} erreurs")

driver.close()
print("\nTerminé !")